# Entity Resonance Quality Check

Validate the NER output from `extract_entities.py`:
- Entity extraction quality (are spaCy Italian entities meaningful?)
- Entity resonance patterns (how many entities appear in both transcripts AND comments?)
- Data integrity (no nulls, expected distributions)

**Files used:**
- `entities_transcripts.parquet` — 11,801 videos
- `entities_comments.parquet` — 12,479 video groups
- `entity_resonance.parquet` — joined (video, entity) pairs

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the three entity files
df_trans = pd.read_parquet('entities_transcripts.parquet')
df_comm  = pd.read_parquet('entities_comments.parquet')
df_res   = pd.read_parquet('entity_resonance.parquet')

print(f"Transcripts: {len(df_trans):,} rows")
print(f"Comments:    {len(df_comm):,} rows")
print(f"Resonance:   {len(df_res):,} rows")
print(f"\nColumns in resonance table:")
print(df_res.columns.tolist())
print(f"\nData types:")
print(df_res.dtypes)

## Entity Statistics

In [ ]:
# Unique entities across the dataset
unique_entities = df_res['entity_text'].nunique()
unique_labels = df_res['entity_label'].nunique()

print(f"Total unique entities:        {unique_entities:,}")
print(f"Total unique entity labels:   {unique_labels:,}")
print(f"\nEntity label distribution:")
print(df_res['entity_label'].value_counts())
print(f"\nTranscript vs Comment presence:")
print(df_res[['in_transcript', 'in_comments']].value_counts())

## Resonance Quality: Entities in Both Transcript AND Comments

In [ ]:
# High-quality resonance: entity appears in transcript AND mentioned in comments
resonant = df_res[df_res['in_transcript'] & df_res['in_comments']]
print(f"Entities appearing in BOTH transcript and comments: {len(resonant):,}")
print(f"  Percentage of all (video, entity) pairs: {len(resonant)/len(df_res)*100:.1f}%")
print(f"\nTop 20 most-commented resonant entities:")
top_resonant = resonant.nlargest(20, 'comment_count')[['entity_text', 'entity_label', 'transcript_count', 'comment_count']]
print(top_resonant.to_string(index=False))

## Sample Quality Inspection: Random Videos

In [ ]:
# Pick 3 random videos and show their top entities
sample_vids = np.random.choice(df_res['video_id'].unique(), size=3, replace=False)

for vid in sample_vids:
    vid_ents = df_res[df_res['video_id'] == vid].nlargest(8, 'comment_count')
    print(f"\n{'='*70}")
    print(f"Video: {vid}")
    print(f"{'='*70}")
    print(vid_ents[['entity_text', 'entity_label', 'transcript_count', 'comment_count', 'in_transcript', 'in_comments']].to_string(index=False))

## Entity Engagement: Most-Commented Entities Across All Videos

In [ ]:
# Aggregate: which entities got the most comment mentions overall?
entity_engagement = df_res.groupby(['entity_text', 'entity_label']).agg({
    'comment_count': 'sum',
    'transcript_count': 'sum',
    'video_id': 'nunique',
}).reset_index()
entity_engagement.columns = ['entity_text', 'entity_label', 'total_comment_mentions', 'total_transcript_mentions', 'num_videos']
entity_engagement = entity_engagement.sort_values('total_comment_mentions', ascending=False)

print(f"Top 20 most-commented entities across all videos:")
print(entity_engagement.head(20).to_string(index=False))

## Potential Issues: Off-Topic Comments (entities in comments but NOT in transcript)

In [ ]:
# These might indicate off-topic discussion or hallucinated entities
off_topic = df_res[~df_res['in_transcript'] & df_res['in_comments']]
print(f"Entities in comments but NOT in transcript: {len(off_topic):,}")
print(f"  Percentage: {len(off_topic)/len(df_res)*100:.1f}%")

print(f"\nTop 15 off-topic entities (might indicate noise or hallucination):")
off_topic_top = off_topic.nlargest(15, 'comment_count')[['entity_text', 'entity_label', 'comment_count', 'video_id']]
print(off_topic_top.to_string(index=False))

## Quality Assessment Summary

In [ ]:
print("=" * 70)
print("ENTITY RESONANCE QUALITY ASSESSMENT")
print("=" * 70)

total_rows = len(df_res)
resonant_rows = len(resonant)
off_topic_rows = len(off_topic)
only_transcript = len(df_res[df_res['in_transcript'] & ~df_res['in_comments']])

print(f"\n1. RESONANCE RATE (entities in both sources):")
print(f"   {resonant_rows:,} / {total_rows:,} = {resonant_rows/total_rows*100:.1f}%")
print(f"   -> {100 - resonant_rows/total_rows*100:.1f}% are partial (one source only)")

print(f"\n2. DATA QUALITY:")
print(f"   Entities in transcript only: {only_transcript:,} ({only_transcript/total_rows*100:.1f}%)")
print(f"   Entities in comments only:   {off_topic_rows:,} ({off_topic_rows/total_rows*100:.1f}%)")
print(f"   -> Low off-topic rate suggests good NER quality")

print(f"\n3. ENTITY LABEL DISTRIBUTION:")
for label in df_res['entity_label'].value_counts().index:
    count = (df_res['entity_label'] == label).sum()
    pct = count / total_rows * 100
    print(f"   {label:8s}: {count:>7,} ({pct:>5.1f}%)")

print(f"\n4. RECOMMENDATION:")
if resonant_rows / total_rows > 0.5:
    print(f"   OK GOOD: {resonant_rows/total_rows*100:.1f}% resonance rate is healthy.")
    print(f"     The entity extraction successfully identifies topics viewers discuss.")
else:
    print(f"   CAUTION: Only {resonant_rows/total_rows*100:.1f}% resonance.")
    print(f"     High rate of one-sided entities may indicate NER noise or viewer off-topic chatter.")

if off_topic_rows / total_rows < 0.1:
    print(f"   OK GOOD: Off-topic rate is low ({off_topic_rows/total_rows*100:.1f}%).")
else:
    print(f"   CAUTION: {off_topic_rows/total_rows*100:.1f}% of entities appear in comments but not transcripts.")

print(f"\n5. NEXT STEPS:")
print(f"   - Proceed to Phase 1 (semantic embedding)")
print(f"   - In Phase 4 (causal inference), use the resonance table to test:")
print(f"     'Do videos discussing topic X (high transcript count) get more comments mentioning X?'")
print(f"   - Filter to resonant entities only (in_transcript=True AND in_comments=True)")
print(f"     for the cleanest causal signal.")

---
## NER Quality Findings Summary

**Model:** `it_core_news_lg` — spaCy Italian large model, trained on news corpora.

### What worked
- Legitimate named entities captured correctly: character names (*Voldemort, Winky, Ludo* in the Harry Potter video), football clubs (*Juve, Inter, Milan, Roma*), real people, zodiac sign content (*Scorpione, Capricorno, Sagittario* dominating comments on astrology videos), brand names (*YouTube, Facebook*).
- PER and ORG are the most trustworthy labels.

### Known weaknesses (expected for this model on informal speech)

| Issue | Examples | Root cause |
|-------|---------|------------|
| Common Italian words tagged as entities | *Riesci* (LOC), *Ciao* (LOC), *Ah* (PER), *Vai* (PER) | Model trained on formal news; informal spoken Italian confuses it |
| MISC is a noise catch-all (49% of all pairs) | *"Domanda mistery"*, *xD*, *Bellissimo*, emojis | MISC has no tight definition; vacuums up comment-section patterns |
| Low apparent resonance (0.9%) | — | Mostly a measurement artifact — comments bring @mentions, emojis, personal references not in transcript |

### Recommended filter for Phase 4 regression
```python
clean = df_res[
    (df_res['entity_label'].isin(['PER', 'ORG'])) &  # drop MISC and noisy LOC
    (df_res['transcript_count'] >= 2) &               # must appear ≥ 2× in transcript
    (df_res['comment_count']    >= 3)                 # must appear ≥ 3× in comments
]
```

### Output files produced
| File | Content |
|------|---------|
| `entities_transcripts.parquet` | Per-video NER from transcript (11,801 rows) |
| `entities_comments.parquet` | Per-video NER aggregated from comments (12,479 rows) |
| `entity_resonance.parquet` | Joined (video, entity) pairs — 2.6 M rows |
| `TargetCreator_entity_dataset.jsonl` | TargetCreator per-video dataset: `{videoId, transcript_entities, comment_entities}` |

---
## Channel Deep-Dive: TargetCreator

Focused entity resonance analysis for the TargetCreator channel, then export a per-video dataset.

In [ ]:
# Load videos parquet to get channelTitle and video metadata
df_videos = pd.read_parquet(
    r'<DATA_DIR>\Data_Cleaned\yt_videos_with_local_transcripts.parquet'
)
cami_ids  = set(df_videos.loc[df_videos['channelTitle'] == 'TargetCreator', 'videoId'])
cami_meta = (
    df_videos.loc[df_videos['channelTitle'] == 'TargetCreator',
                  ['videoId', 'title', 'is_short', 'viewCount', 'commentCount']]
    .copy()
)
print(f"TargetCreator videos in dataset : {len(cami_ids):,}")
print(f"  shorts : {cami_meta['is_short'].sum():,}")
print(f"  long   : {(~cami_meta['is_short']).sum():,}")
print(f"  avg views      : {pd.to_numeric(cami_meta['viewCount'], errors='coerce').mean():,.0f}")
print(f"  avg comments   : {pd.to_numeric(cami_meta['commentCount'], errors='coerce').mean():,.1f}")

In [ ]:
# TargetCreator resonance subset
cami_res = df_res[df_res['video_id'].isin(cami_ids)].copy()
cami_resonant = cami_res[cami_res['in_transcript'] & cami_res['in_comments']]

print(f"(video, entity) pairs for TargetCreator : {len(cami_res):,}")
print(f"  Resonant (both sources)            : {len(cami_resonant):,}  "
      f"({len(cami_resonant)/max(len(cami_res),1)*100:.1f}%)")
print()

print("── Top 20 entities by transcript mentions (TargetCreator) ──")
top_t = (
    cami_res[cami_res['in_transcript']]
    .groupby(['entity_text', 'entity_label'])['transcript_count']
    .sum().reset_index()
    .nlargest(20, 'transcript_count')
)
print(top_t.to_string(index=False))

print()
print("── Top 20 entities by comment mentions (TargetCreator) ──")
top_c = (
    cami_res[cami_res['in_comments']]
    .groupby(['entity_text', 'entity_label'])['comment_count']
    .sum().reset_index()
    .nlargest(20, 'comment_count')
)
print(top_c.to_string(index=False))

In [ ]:
import json

# Build per-video entity dataset for TargetCreator.
# Each record: videoId, title, is_short, viewCount, commentCount,
#              transcript_entities {text: count}, comment_entities {text: count}

meta_lookup = cami_meta.set_index('videoId')

records = []
for vid in sorted(cami_ids):
    t_rows = cami_res[(cami_res['video_id'] == vid) & cami_res['in_transcript']]
    c_rows = cami_res[(cami_res['video_id'] == vid) & cami_res['in_comments']]

    transcript_ents = dict(zip(t_rows['entity_text'], t_rows['transcript_count'].astype(int)))
    comment_ents    = dict(zip(c_rows['entity_text'], c_rows['comment_count'].astype(int)))

    # Sort each dict by count descending for readability
    transcript_ents = dict(sorted(transcript_ents.items(), key=lambda x: -x[1]))
    comment_ents    = dict(sorted(comment_ents.items(),    key=lambda x: -x[1]))

    row = meta_lookup.loc[vid] if vid in meta_lookup.index else None
    records.append({
        'videoId':              vid,
        'title':                str(row['title'])        if row is not None else '',
        'is_short':             bool(row['is_short'])    if row is not None else None,
        'viewCount':            int(pd.to_numeric(row['viewCount'],   errors='coerce') or 0)
                                if row is not None else 0,
        'commentCount':         int(pd.to_numeric(row['commentCount'], errors='coerce') or 0)
                                if row is not None else 0,
        'transcript_entities':  transcript_ents,
        'comment_entities':     comment_ents,
    })

out_path = 'TargetCreator_entity_dataset.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f"Saved {len(records):,} video records -> {out_path}")
print()
# Preview two records
for rec in records[:2]:
    print(f"videoId: {rec['videoId']}")
    print(f"  title        : {rec['title'][:70]}")
    print(f"  is_short     : {rec['is_short']}   views: {rec['viewCount']:,}   comments: {rec['commentCount']:,}")
    print(f"  transcript_entities (top 5): { dict(list(rec['transcript_entities'].items())[:5]) }")
    print(f"  comment_entities    (top 5): { dict(list(rec['comment_entities'].items())[:5]) }")
    print()